In [ ]:
# ==============================================================================
# English Company Review Analysis Pipeline (Pros + Cons)
# ==============================================================================

# 1. Install libraries
print("1. Installing libraries...")
!pip install emoji torch scipy pandas openpyxl tqdm matplotlib datasets scikit-learn transformers bertopic sentence-transformers umap-learn --upgrade --quiet

# ==============================================================================
# 1.필요 라이브러리 설치
# ==============================================================================
print("2. Importing libraries...")

import pandas as pd
import numpy as np
import re, html
import torch
from math import log
from tqdm import tqdm
from google.colab import files

import plotly.io as pio
pio.renderers.default = "colab"

import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

import matplotlib.pyplot as plt
import seaborn as sns

import gc
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
from networkx.algorithms import community

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"-> Device: {device}")

# ==============================================================================
# 2. 리뷰 전처리(중복되는 단어 위주로)
# ==============================================================================

nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

custom_remove = {
    "notion", "clickup", "chatgpt", "also", "thats",
    "use", "used", "using", "easy", "like", "feature", "features",
    "tool", "tools", "page", "pages", "good", "great", "really",
    "app", "application", "software", "get", "even", "bit"
}

html_tag_pattern = re.compile(r"<.*?>")
url_pattern = re.compile(r"https?:\/\/\S+")
special_pattern = re.compile(r"[^a-zA-Z\s]")

def clean_english(text):
    text = str(text).lower()
    text = html.unescape(text)
    text = html_tag_pattern.sub("", text)
    text = url_pattern.sub("", text)
    text = special_pattern.sub(" ", text)

    tokens = []
    for w in text.split():
        if (
            w not in stop_words
            and w not in custom_remove
            and len(w) > 2
            and w.isalpha()
        ):
            #단어를 기본형(단수형 등)으로 변환
            # 예: databases -> database, apples -> apple
            lemma_w = lemmatizer.lemmatize(w)
            tokens.append(lemma_w)

    return " ".join(tokens)

# ==============================================================================
# 3. 엑셀 로드 및 전처리 진행
# ==============================================================================

print("\n--- Upload Excel File ---")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)

df.columns = [c.strip() for c in df.columns]

required_cols = [
    "Company", "ID", "Name", "JobRole", "CompanySize",
    "ReviewDate", "Rating", "Pros", "Cons"
]

for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"Missing column: {c}")

# Combine Pros + Cons
df["Review_Raw"] = df["Pros"].fillna("") + " " + df["Cons"].fillna("")
df["Review_Cleaned"] = df["Review_Raw"].apply(clean_english)

df = df[df["Review_Cleaned"].str.len() > 20].copy()
print(f"-> Analysis samples: {len(df)}")

# ==============================================================================
# 4. 기업 크기 숫자와(small = 1, mid =2, enterprise =3)
# ==============================================================================

def encode_company_size(row):
    text = f"{row['JobRole']} {row['CompanySize']}".lower()
    if "small-business" in text:
        return 1
    if "mid-market" in text:
        return 2
    if "enterprise" in text:
        return 3
    return np.nan

df["companysize_num"] = df.apply(encode_company_size, axis=1)

# ==============================================================================
# 5.점수 /5 제거
# ==============================================================================

def normalize_rating(x):
    try:
        return float(str(x).split("/")[0])
    except:
        return np.nan

df["rating_num"] = df["Rating"].apply(normalize_rating)

# ==============================================================================
# 6. Word Count
# ==============================================================================

df["word_count"] = df["Review_Cleaned"].apply(lambda x: len(x.split()))
df["word_count_log"] = df["word_count"].apply(lambda x: log(x + 1))

# ==============================================================================
# 7. Subjectivity
# ==============================================================================

subjectivity_clf = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if torch.cuda.is_available() else -1
)

def get_subjectivity(texts):
    results = subjectivity_clf(texts, ["opinion", "fact"])
    return [r["scores"][r["labels"].index("opinion")] for r in results]

subs = []
#전처리를 bertopic을 위해 좀 빡세게 해서 원본 리뷰로
texts_for_analysis = df["Review_Raw"].tolist()

for i in tqdm(range(0, len(texts_for_analysis), 32)):
    subs.extend(get_subjectivity(texts_for_analysis[i:i+32]))

df["subjectivity_score"] = subs

# ==============================================================================
# 8. Sentiment
# ==============================================================================

MODEL_SENT = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL_SENT)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_SENT).to(device)

def sentiment_scores(texts):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim=1).cpu().numpy()
    return [round(p[2] - p[0], 4) for p in probs]

sent_scores = []
for i in tqdm(range(0, len(texts_for_analysis), 32)):
    sent_scores.extend(sentiment_scores(texts_for_analysis[i:i+32]))

df["sentiment_score"] = sent_scores

# ==============================================================================
# 9. BERTopic
# ==============================================================================

print("\n--- BERTopic Analysis ---")
embedding_model = SentenceTransformer("all-mpnet-base-v2")
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=8, min_samples=5, metric='euclidean', prediction_data=True)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,
    verbose=True
)

docs = df["Review_Cleaned"].tolist()
topics, _ = topic_model.fit_transform(docs)
#Outlier Reduction 실행 (이게 -1을 다른 토픽으로 흡수시켜줌)
print("-> Checking for outliers...")
if -1 in topics:
    print("-> Outliers found. Reducing outliers...")
    new_topics = topic_model.reduce_outliers(docs, topics, strategy="embeddings")
    topic_model.update_topics(docs, topics=new_topics)
    # 업데이트된 토픽을 저장
    df["bertopic_topic_id"] = new_topics
else:
    print("-> No outliers found (-1 topic is empty). Skipping reduction.")
    # 이상치가 없으므로 기존 topics를 그대로 사용
    df["bertopic_topic_id"] = topics

# 업데이트된 토픽을 저장
df["bertopic_topic_id"] = new_topics

# ==============================================================================
# 9-1. Meta-topic (Louvain Clustering 적용)
# ==============================================================================

print("\n--- Applying Optimized Louvain Clustering ---")

# 1. BERTopic이 이미 계산한 토픽 임베딩 추출
all_topic_embeddings = topic_model.topic_embeddings_
topic_info = topic_model.get_topic_info()
topic_ids = sorted(topic_info[topic_info["Topic"] != -1]["Topic"].tolist())

# Outlier(-1)을 제외한 실제 토픽 인덱싱 (BERTopic의 인덱스는 Topic ID + 1)
actual_indices = [tid + 1 for tid in topic_ids]
relevant_embeddings = all_topic_embeddings[actual_indices]

# 2. 코사인 유사도 계산 (시각화에서 사용해야 하므로 여기서 del 하지 않음)
sim_matrix = cosine_similarity(relevant_embeddings)

# 3. Graph 생성 및 Louvain 적용
G = nx.Graph()
#0.7로 하니까 큰거 3개에 나머지 메타토픽들이 거길 중심으로 존재함
threshold = 0.5

for i in range(len(topic_ids)):
    for j in range(i + 1, len(topic_ids)):
        weight = sim_matrix[i][j]
        if weight > threshold:
            G.add_edge(topic_ids[i], topic_ids[j], weight=weight)

comm_sets = community.louvain_communities(G, weight='weight', seed=42)

# 4. 결과 매핑
meta_map = {}
for meta_idx, nodes in enumerate(comm_sets):
    for node in nodes:
        meta_map[node] = meta_idx

current_max_meta = len(comm_sets)
for tid in topic_ids:
    if tid not in meta_map:
        meta_map[tid] = current_max_meta
        current_max_meta += 1

df["meta_topic_id"] = df["bertopic_topic_id"].map(meta_map).fillna(-1)

print("-> Meta-topic Clustering Mapping Completed.")

# ==============================================================================
# 9-2. Meta-topic Mapping Excel 생성
# ==============================================================================
print("\n--- Generating Meta-topic Mapping Table (Original -> Meta) ---")

rows = []
# 1. 병합 전의 원래 토픽 ID들을 순회합니다.
for tid in topic_ids:
    meta_id = meta_map.get(tid)

    # 해당 토픽의 데이터 추출
    sample_data = df[df["bertopic_topic_id"] == tid]

    if not sample_data.empty:
        # 원래 토픽의 대표 리뷰 1개
        sample_review = sample_data["Review_Cleaned"].iloc[0]

        # 원래 토픽의 키워드 (모델 병합 전이라 정상 호출됨)
        raw_keywords = topic_model.get_topic(tid)
        if raw_keywords:
            keywords_str = ", ".join([f"{w}:{round(s,3)}" for w, s in raw_keywords])

            # [메타 ID, 원래 토픽 ID, 원래 키워드, 대표 리뷰] 저장
            rows.append([meta_id, tid, keywords_str, sample_review])

# 데이터프레임 생성 및 정렬 (메타 ID 기준)
meta_df = pd.DataFrame(
    rows,
    columns=["Meta_Topic_ID", "Original_Topic_ID", "Original_Keywords", "Representative_Review"]
).sort_values(by=["Meta_Topic_ID", "Original_Topic_ID"])

# 엑셀 파일 다운로드
meta_output = filename.replace(".xlsx", "_Detailed_Topic_Mapping.xlsx")
meta_df.to_excel(meta_output, index=False)
files.download(meta_output)

print(f"-> Mapping table saved: {meta_output}")

# ==============================================================================
# 9-3. Metatopic으로 그룹화
# ==============================================================================

print("\n--- Updating Model to Meta-topics for Better Visualization ---")

# 1. 루뱅 결과(meta_map)를 바탕으로 병합할 리스트 생성
# 같은 meta_topic_id를 가진 원래 토픽들을 그룹화합니다.
meta_to_original = {}
for t_id, m_id in meta_map.items():
    if m_id not in meta_to_original:
        meta_to_original[m_id] = []
    meta_to_original[m_id].append(t_id)

# 2개 이상의 토픽이 모인 그룹만 병합 대상으로 지정
topics_to_merge = [ids for ids in meta_to_original.values() if len(ids) > 1]

# 2. 모델 업데이트 (모든 시각화가 이제 메타 토픽 기준으로 바뀝니다)
if topics_to_merge:
    topic_model.merge_topics(docs, topics_to_merge)
    print(f"-> Successfully merged into {len(meta_to_original)} Meta-topics.")

# --- 이제부터 실행되는 모든 시각화(Intertopic, Barchart 등)는 메타 토픽 기준으로 나옵니다 ---

# ==============================================================================
# 9-4. MetaTopic 시각화
# ==============================================================================
print("\n--- MetaTopic Visualizations ---")

topic_info = topic_model.get_topic_info()
# -1(Outlier)을 제외한 실제 토픽들
valid_topics_df = topic_info[topic_info["Topic"] != -1]
num_valid_topics = len(valid_topics_df)

print(f"-> Total topics found (excluding outliers): {num_valid_topics}")

# (1) Intertopic Distance Map
if num_valid_topics >= 2:
    try:
        print("Generating Intertopic Distance Map...")
        fig1 = topic_model.visualize_topics()
        fig1.show(renderer="colab")
        fig1.write_html("1_intertopic_distance_map.html") # 파일로도 저장
    except Exception as e: print(f"[ERR] visualize_topics failed: {e}")
else:
    print("[SKIP] Intertopic Distance Map: Need at least 2 valid topics.")

# (2) Hierarchical Clustering
if num_valid_topics >= 2:
    try:
        print("Generating Hierarchical Clustering Chart...")
        fig2 = topic_model.visualize_hierarchy()
        fig2.show(renderer="colab")
        fig2.write_html("2_hierarchical_clustering.html")
    except Exception as e: print(f"[ERR] visualize_hierarchy failed: {e}")
else:
    print("[SKIP] Hierarchical Clustering: Need at least 2 valid topics.")

# (3) Topic Word Scores (Barchart)
if num_valid_topics >= 1:
    try:
        print("Generating Topic Word Scores Barchart...")
        # 상위 10개 혹은 전체 유효 토픽 개수만큼 표시
        fig3 = topic_model.visualize_barchart(top_n_topics=min(10, num_valid_topics))
        fig3.show(renderer="colab")
        fig3.write_html("3_topic_word_scores.html")
    except Exception as e: print(f"[ERR] visualize_barchart failed: {e}")
else:
    print("[SKIP] Topic Word Scores: No valid topics found.")

# (4) Topic Similarity Heatmap (Louvain 기반)
if len(topic_ids) > 0:
    try:
        print("Generating Similarity Heatmap...")
        plt.figure(figsize=(12, 10))
        sns.heatmap(sim_matrix, annot=True, cmap="coolwarm",
                    xticklabels=topic_ids, yticklabels=topic_ids)
        plt.title("Meta-Topic Similarity Matrix (Louvain Source)")
        plt.show()
        plt.close()
    except Exception as e: print(f"[ERR] Heatmap failed: {e}")

# 시각화 파일 다운로드 안내
print("\n-> Visualization HTML files have been saved in the sidebar folder.")


# ==============================================================================
# 10. Save Main Output
# ==============================================================================

output_file = filename.replace(".xlsx", "_CompanyReview_Analyzed.xlsx")
df.to_excel(output_file, index=False)
files.download(output_file)


1. Installing libraries...
2. Importing libraries...
-> Device: cuda


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...



--- Upload Excel File ---


[nltk_data]   Package omw-1.4 is already up-to-date!


Saving g2-clickup.xlsx to g2-clickup.xlsx
-> Analysis samples: 9450


Device set to use cuda:0
100%|██████████| 296/296 [09:10<00:00,  1.86s/it]
Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
100%|██████████| 296/296 [01:37<00:00,  3.04it/s]


... (2 outputs left out)

Batches:   0%|          | 0/296 [00:00<?, ?it/s]

2026-01-02 05:05:32,127 - BERTopic - Embedding - Completed ✓
2026-01-02 05:05:32,128 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-02 05:05:53,080 - BERTopic - Dimensionality - Completed ✓
2026-01-02 05:05:53,081 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-02 05:05:54,032 - BERTopic - Cluster - Completed ✓
2026-01-02 05:05:54,038 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-02 05:05:54,211 - BERTopic - Representation - Completed ✓


-> Reducing outliers...


ValueError: No outliers to reduce.

위에서 나온 결과 엑셀 파일에서 big5 점수 도출 코드

In [ ]:
!pip install transformers pandas tqdm openpyxl --quiet

import pandas as pd
from transformers import pipeline
from tqdm import tqdm

# ===============================
# 1. 결과 엑셀 로드
# ===============================
INPUT_FILE = "g2-clickup_CompanyReview_Analyzed.xlsx"   # ← 결과로 나오는 파일명과 일치 시키기
OUTPUT_FILE = "_with_big5.xlsx"

df = pd.read_excel(INPUT_FILE)

if "Review_Cleaned" not in df.columns:
    raise ValueError("Review_Cleaned 컬럼이 없습니다")

# ===============================
# 2. Zero-shot 모델 로드 (1회)
# ===============================
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0  # GPU 없으면 제거
)

# ===============================
# 3. Big5 라벨
# ===============================
BIG5_LABELS = {
    "Openness": [
        ("Very Conservative", 0.0),
        ("A bit Conservative", 0.25),
        ("Neutral", 0.5),
        ("A bit Open", 0.75),
        ("Very Open", 1.0),
    ],
    "Conscientiousness": [
        ("Very Careless", 0.0),
        ("A bit Careless", 0.25),
        ("Neutral", 0.5),
        ("A bit Conscientious", 0.75),
        ("Very Conscientious", 1.0),
    ],
    "Extraversion": [
        ("Very Introverted", 0.0),
        ("A bit Introverted", 0.25),
        ("Neutral", 0.5),
        ("A bit Extraverted", 0.75),
        ("Very Extraverted", 1.0),
    ],
    "Agreeableness": [
        ("Very Hostile", 0.0),
        ("A bit Hostile", 0.25),
        ("Neutral", 0.5),
        ("A bit Agreeable", 0.75),
        ("Very Agreeable", 1.0),
    ],
    "Neuroticism": [
      ("Very Emotionally Stable", 0.0),
      ("A bit Emotionally Stable", 0.25),
      ("Neutral", 0.5),
      ("A bit Neurotic", 0.75),
      ("Very Neurotic", 1.0)
    ],
}

# ===============================
# 4. Trait 설명 (프롬프트 분리용)
# ===============================
TRAIT_DESCRIPTIONS = {
    "Openness": "openness to experience",
    "Conscientiousness": "conscientiousness",
    "Extraversion": "extraversion",
    "Agreeableness": "agreeableness",
    "Neuroticism": "neuroticism",
}

# ===============================
# 5. 점수 계산 함수
# ===============================
def score_big5(text, trait, labels):
    if pd.isna(text) or not str(text).strip():
        return 0.0

    label_names = [l for l, _ in labels]

    output = classifier(
        text,
        candidate_labels=label_names,
        hypothesis_template=f"The author shows {{}} level of {TRAIT_DESCRIPTIONS[trait]}.",
        return_all_scores=True
    )

    # 출력 구조 안전 처리
    if isinstance(output, list):
        result = output[0]
    else:
        result = output

    score = sum(
        s * w
        for s, (_, w) in zip(result["scores"], labels)
    )

    return round(score, 4)

# ===============================
# 6. Big5 전체 계산
# ===============================
tqdm.pandas(desc="Big5 Scoring")

for trait, labels in BIG5_LABELS.items():
    df[f"{trait}_score"] = df["Review_Cleaned"].progress_apply(
        lambda x: score_big5(x, trait, labels)
    )

# ===============================
# 7. 저장
# ===============================
df.to_excel(OUTPUT_FILE, index=False)
print(f"저장 완료: {OUTPUT_FILE}")

Device set to use cuda:0
Big5 Scoring: 100%|██████████| 9459/9459 [17:29<00:00,  9.01it/s]


저장 완료: _with_big5.xlsx


metatopic 점수

In [ ]:
import pandas as pd
import torch
from transformers import pipeline
from tqdm import tqdm
from google.colab import files

# ==============================================================================
# 1. 브랜드별 라벨 정의
# ==============================================================================
NOTION_LABELS = [
    "Learning Curve and Ease of Adoption", "Information Search and Content Retrieval",
    "AI-Assisted Writing and Content Creation", "Note-Taking and Knowledge Capture",
    "All-in-One Workspace and Productivity Hub", "Team Collaboration and Shared Workspaces",
    "Cross-Device and Platform Accessibility", "Personal Organization and Life Management",
    "Business and Professional Use Cases", "Overall User Satisfaction and Perceived Value"
]

CLICKUP_LABELS = [
    "Business-Oriented Work Management", "User Interface and Usability Experience",
    "Task Views, Performance, and System Speed", "Time Tracking and Task Accountability",
    "Project and Task Management Core Functions", "Pricing, Plans, and Cost Effectiveness",
    "Mobile App and Cross-Platform Usage", "Team Collaboration and Work Coordination",
    "Integrations and In-App Communication"
]

TARGET_BRAND = "clickup"  # 분석하려는 브랜드에 맞춰 "notion" 또는 "clickup"으로 변경
current_labels = NOTION_LABELS if TARGET_BRAND == "notion" else CLICKUP_LABELS

# ==============================================================================
# 2. 파일 로드 및 모델 준비
# ==============================================================================
print(f"--- Upload Analysis File for {TARGET_BRAND.upper()} ---")
uploaded = files.upload()
main_file = list(uploaded.keys())[0]
df = pd.read_excel(main_file)

# GPU 가속 설정
device = 0 if torch.cuda.is_available() else -1
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device)

# ==============================================================================
# 3. 라벨별 개별 점수 산출 (배치 처리 최적화)
# ==============================================================================
print(f"\n--- Calculating Scores for each of the {len(current_labels)} Labels ---")

def get_label_scores_only(texts, labels):
    all_results = []
    batch_size = 16 # GPU 메모리에 따라 16~32 권장

    for i in tqdm(range(0, len(texts), batch_size), desc="Processing Batches"):
        batch_texts = texts[i:i+batch_size]
        results = classifier(
            batch_texts,
            candidate_labels=labels,
            hypothesis_template="This review is about {}.",
            multi_label=True # 각 라벨에 대한 독립적인 확률 점수를 얻기 위해 True 설정
        )

        for res in results:
            # {라벨명: 점수} 형태로 매핑하여 리스트에 추가
            label_scores = {f"score_{l}": round(s, 4) for l, s in zip(res['labels'], res['scores'])}
            all_results.append(label_scores)

    return all_results

# 텍스트 데이터 준비 (리뷰 컬럼명이 Review_Cleaned인지 확인하세요)
texts_to_analyze = df['Review_Cleaned'].fillna("").astype(str).tolist()

# 분석 실행
score_data = get_label_scores_only(texts_to_analyze, current_labels)
score_df = pd.DataFrame(score_data)

# 기존 데이터프레임과 병합 (원본 데이터 옆에 점수 컬럼들이 붙음)
df = pd.concat([df.reset_index(drop=True), score_df], axis=1)

# ==============================================================================
# 4. 저장 및 다운로드
# ==============================================================================
output_name = f"{TARGET_BRAND}_Label_Scores_Analyzed.xlsx"
df.to_excel(output_name, index=False)
files.download(output_name)

print(f"\n완료! {len(current_labels)}개의 점수 컬럼이 생성되었습니다.")

--- Upload Analysis File for CLICKUP ---


Saving g2-clickup_CompanyReview_Analyzed_with_big5.xlsx to g2-clickup_CompanyReview_Analyzed_with_big5.xlsx


Device set to use cuda:0



--- Calculating Scores for each of the 9 Labels ---


Processing Batches: 100%|██████████| 592/592 [30:33<00:00,  3.10s/it]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


완료! 9개의 점수 컬럼이 생성되었습니다.


Humanlikeness 점수

In [ ]:
import pandas as pd
import numpy as np
import torch  # GPU 체크를 위해 추가
from transformers import pipeline
from tqdm import tqdm
from google.colab import files

# --------------------------------------------------
# 1. 파일 업로드
# --------------------------------------------------
print("--- Humanlikeness 점수를 생성할 엑셀 파일을 업로드하세요 ---")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)

# Review_Raw 컬럼 존재 여부 확인
if 'Review_Raw' not in df.columns:
    print("에러: 엑셀 파일에 'Review_Raw' 컬럼이 없습니다.")
else:
    # --------------------------------------------------
    # 2. 모델 로드 및 GPU 설정 (수정됨)
    # --------------------------------------------------
    # PyTorch를 사용하여 실제 GPU 사용 가능 여부 확인
    device = 0 if torch.cuda.is_available() else -1
    print(f"\n--- BART 모델 로딩 중 (사용 기기: {'GPU' if device == 0 else 'CPU'}) ---")

    classifier = pipeline(
        "zero-shot-classification",
        model="facebook/bart-large-mnli",
        device=device
    )

    humanlikeness_labels = [
        ("Very Machianlike", 0.0),
        ("Somewhat Machianlike", 0.25),
        ("Neutral", 0.5),
        ("Somewhat Humanlike", 0.75),
        ("Very Humanlike", 1.0)
    ]

    def classify_humanlikeness_score(comment):
        if pd.isna(comment) or not str(comment).strip():
            return 0.5

        labels = [label for label, _ in humanlikeness_labels]

        try:
            # truncation=True를 통해 모델 최대 길이(1024 토큰)를 초과하는 리뷰 자동 절단
            output = classifier(
                str(comment),
                candidate_labels=labels,
                hypothesis_template="This comment mention to {} in terms of uncanny valley theory in conversational AI.",
                truncation=True  # 긴 문장 처리 핵심 옵션
            )

            result_scores = {label: score for label, score in zip(output['labels'], output['scores'])}
            weighted_sum = sum(result_scores[label] * weight for label, weight in humanlikeness_labels)
            return round(weighted_sum, 4)

        except Exception as e:
            # 에러 발생 시 로그를 남기고 중립 점수(0.5) 반환
            return 0.5

    # --------------------------------------------------
    # 3. 실행 및 결과 다운로드
    # --------------------------------------------------
    print(f"\n--- 총 {len(df)}건 분석 시작 (긴 리뷰 자동 절단 적용) ---")
    tqdm.pandas()
    df["Humanlikeness_score"] = df["Review_Raw"].progress_apply(classify_humanlikeness_score)

    output_file = filename.replace(".xlsx", "_with_humanlikeness.xlsx")
    df.to_excel(output_file, index=False)

    print(f"\n완료! 파일 다운로드 중: {output_file}")
    files.download(output_file)

--- Humanlikeness 점수를 생성할 엑셀 파일을 업로드하세요 ---


Saving 1.clickup_Label_Scores_Analyzed.xlsx to 1.clickup_Label_Scores_Analyzed.xlsx

--- BART 모델 로딩 중 (사용 기기: GPU) ---


Device set to use cuda:0



--- 총 9459건 분석 시작 (긴 리뷰 자동 절단 적용) ---


100%|██████████| 9459/9459 [25:07<00:00,  6.28it/s]



완료! 파일 다운로드 중: 1.clickup_Label_Scores_Analyzed_with_humanlikeness.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

correlation and descriptive statistics

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
from google.colab import files

# --------------------------------------------------
# 1. 파일 업로드
# --------------------------------------------------
print("--- 분석할 엑셀 파일을 업로드하세요 ---")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)

# --------------------------------------------------
# 2. 기업 규모 더미 생성
# --------------------------------------------------
df["companysize_cat"] = df["companysize_num"].map({
    1: "Small",
    2: "Medium",
    3: "Large"
}).fillna("Unknown")

company_dummies = pd.get_dummies(
    df["companysize_cat"],
    prefix="CompanySize",
    drop_first=True
)

df = pd.concat([df, company_dummies], axis=1)
company_vars = list(company_dummies.columns)

# --------------------------------------------------
# 3. 변수 자동 감지 (회귀 코드와 동일한 로직)
# --------------------------------------------------

# Big5
big5_candidates = [
    'Openness_score',
    'Conscientiousness_score',
    'Extraversion_score',
    'Agreeableness_score',
    'Neuroticism_score'
]
big5 = [c for c in big5_candidates if c in df.columns]

# Moderator: humanlikeness 있으면 그거, 없으면 sentiment_score
if 'Humanlikeness_score' in df.columns:
    moderator = 'Humanlikeness_score'
    print("humanlikeness 변수를 상관분석에 포함합니다.")
else:
    moderator = 'sentiment_score'
    print("Humanlikeness_score 없음 → sentiment_score로 대체")

# Controls
controls_candidates = [
    'sentiment_score'
    #'subjectivity_score',
    #'word_count_log'
]
controls = [c for c in controls_candidates if c in df.columns]

# 종속변수
dependent = 'rating_num'

# --------------------------------------------------
# 4. 전체 변수 리스트 (회귀와 동일한 구성)
# --------------------------------------------------
all_vars = big5 + [moderator] + controls + company_vars + [dependent]

print(f"\n분석 변수 개수: {len(all_vars)}")
print(all_vars)

# --------------------------------------------------
# 5. 결측치 제거
# --------------------------------------------------
df_clean = df[all_vars].replace([np.inf, -np.inf], np.nan).dropna()

# --------------------------------------------------
# 6. 기술통계
# --------------------------------------------------
desc_stats = df_clean.describe().T[['mean', 'std']]
desc_stats.columns = ['Mean', 'SD']
desc_stats.to_excel("Table1_Descriptive_Statistics.xlsx")

# --------------------------------------------------
# 7. 상관계수 + p-value
# --------------------------------------------------
corr_matrix = pd.DataFrame(index=all_vars, columns=all_vars, dtype=float)
pval_matrix = pd.DataFrame(index=all_vars, columns=all_vars, dtype=float)

for i in all_vars:
    for j in all_vars:
        valid = df_clean[[i, j]].dropna()
        if len(valid) < 3:
            continue

        x = np.asarray(valid[i], dtype=float).ravel()
        y = np.asarray(valid[j], dtype=float).ravel()

        r, p = pearsonr(x, y)
        corr_matrix.loc[i, j] = r
        pval_matrix.loc[i, j] = p

# --------------------------------------------------
# 8. 별표 붙인 논문용 표
# --------------------------------------------------
def corr_star(r, p):
    if pd.isna(r): return ""
    if p < 0.01: return f"{r:.3f}**"
    elif p < 0.05: return f"{r:.3f}*"
    else: return f"{r:.3f}"

corr_table = pd.DataFrame("", index=all_vars, columns=all_vars)

for i in range(len(all_vars)):
    for j in range(len(all_vars)):
        if i == j:
            corr_table.iloc[i, j] = "1"
        elif i > j:
            corr_table.iloc[i, j] = corr_star(
                corr_matrix.iloc[i, j],
                pval_matrix.iloc[i, j]
            )

corr_table.to_excel("Table2_Correlation_Matrix.xlsx")

# --------------------------------------------------
# 9. |r| ≥ 0.7 체크
# --------------------------------------------------
high_corr = []
for i in range(len(all_vars)):
    for j in range(i):
        r = corr_matrix.iloc[i, j]
        if pd.notna(r) and abs(r) >= 0.7:
            high_corr.append({
                "Var1": all_vars[i],
                "Var2": all_vars[j],
                "r": round(r, 3)
            })

if high_corr:
    pd.DataFrame(high_corr).to_excel("High_Correlation_Pairs.xlsx", index=False)

# --------------------------------------------------
# 10. 다운로드
# --------------------------------------------------
files.download("Table1_Descriptive_Statistics.xlsx")
files.download("Table2_Correlation_Matrix.xlsx")
if high_corr:
    files.download("High_Correlation_Pairs.xlsx")

print(f"\n최종 샘플 수: {len(df_clean)}")


--- 분석할 엑셀 파일을 업로드하세요 ---


Saving 1.clickup_Label_Scores_Analyzed.xlsx to 1.clickup_Label_Scores_Analyzed.xlsx
humanlikeness 변수를 상관분석에 포함합니다.

분석 변수 개수: 11
['Openness_score', 'Conscientiousness_score', 'Extraversion_score', 'Agreeableness_score', 'Neuroticism_score', 'Humanlikeness_score', 'sentiment_score', 'CompanySize_Medium', 'CompanySize_Small', 'CompanySize_Unknown', 'rating_num']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


최종 샘플 수: 9457


Multiple Regression

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from google.colab import files

# --------------------------------------------------
# 1. 파일 업로드
# --------------------------------------------------
print("--- 분석용 엑셀 파일 업로드 ---")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)

print(f"원본 데이터 개수: {len(df)}")

# --------------------------------------------------
# 2. 변수 설정
# --------------------------------------------------
dependent = 'rating_num'

# Big5
big5_cols = [
    'Openness_score',
    'Conscientiousness_score',
    'Extraversion_score',
    'Agreeableness_score',
    'Neuroticism_score'
]
big5 = [c for c in big5_cols if c in df.columns]

# Moderator
if 'Humanlikeness_score' in df.columns:
    moderator = 'Humanlikeness_score'
    print("Moderator: Humanlikeness_score")
else:
    moderator = 'sentiment_score'
    print("Humanlikeness 없어 sentiment_score를 moderator로 사용")

# Controls
controls = [c for c in ['sentiment_score'] if c in df.columns and c != moderator] #,'subjectivity_score','word_count_log'

print("Big5:", big5)
print("Moderator:", moderator)
print("Controls:", controls)

# --------------------------------------------------
# 3. 결측치 처리 (진짜 쓰는 변수만 기준)
# --------------------------------------------------
used_vars = big5 + [moderator] + controls + [dependent]

df = df.replace([np.inf, -np.inf], np.nan)

print("결측 제거 전:", len(df))

df = df.dropna(subset=used_vars)

print("결측 제거 후:", len(df))

# --------------------------------------------------
# 4. Big5 VIF 완화용 직교화 (선택적이지만 유지)
# --------------------------------------------------
def adjust_vif_for_analysis(df_in, cols, strength=0.98):
    df_out = df_in.copy()
    for i in range(1, len(cols)):
        target = cols[i]
        refs = cols[:i]
        X_tmp = sm.add_constant(df_out[refs])
        model_tmp = sm.OLS(df_out[target], X_tmp).fit()
        df_out[target] = df_out[target] - (strength * model_tmp.predict(X_tmp))
        df_out[target] = df_out[target] - df_out[target].mean() + df_in[target].mean()
        df_out[target] = df_out[target].clip(0, 1)
    return df_out

df_adj = adjust_vif_for_analysis(df, big5)

# --------------------------------------------------
# 5. 평균중심화 & 상호작용항 생성
# --------------------------------------------------
df_mod = df_adj.copy()

for col in big5 + [moderator]:
    df_mod[col + "_centered"] = df_mod[col] - df_mod[col].mean()

interaction_cols = []
for col in big5:
    inter = f"inter_{col}"
    df_mod[inter] = df_mod[col + "_centered"] * df_mod[moderator + "_centered"]
    interaction_cols.append(inter)

# --------------------------------------------------
# 6. 회귀분석
# --------------------------------------------------
X_list = (
    [c + "_centered" for c in big5] +
    [moderator + "_centered"] +
    controls +
    interaction_cols
)

X = df_mod[X_list].astype(float)
y = df_mod[dependent].astype(float)

X_const = sm.add_constant(X)
model = sm.OLS(y, X_const).fit()

# --------------------------------------------------
# 7. VIF 계산
# --------------------------------------------------
vif_map = {}
for i, col in enumerate(X.columns):
    vif_map[col] = variance_inflation_factor(X.values, i)

# --------------------------------------------------
# 8. 결과 저장
# --------------------------------------------------
def get_star(p):
    if p < 0.01: return "***"
    elif p < 0.05: return "**"
    elif p < 0.1: return "*"
    else: return ""

results = pd.DataFrame({
    "Variable": model.params.index,
    "Coefficient": model.params.values,
    "Std.Error": model.bse.values,
    "t-value": model.tvalues.values,
    "p-value": model.pvalues.values,
    "Sig": [get_star(p) for p in model.pvalues],
    "VIF": [vif_map.get(col, np.nan) for col in model.params.index]
})

results["Adjusted R2"] = round(model.rsquared_adj, 4)

results.to_excel("Final_Regression_Result.xlsx", index=False)
files.download("Final_Regression_Result.xlsx")

# --------------------------------------------------
# 9. 출력
# --------------------------------------------------
print(model.summary())
print(f"\n최종 분석 데이터 수: {len(df_mod)}")
print(f"최대 VIF: {max(vif_map.values()):.2f}")


--- 분석용 엑셀 파일 업로드 ---


Saving 1.clickup_Label_Scores_Analyzed.xlsx to 1.clickup_Label_Scores_Analyzed (1).xlsx
원본 데이터 개수: 9459
Moderator: Humanlikeness_score
Big5: ['Openness_score', 'Conscientiousness_score', 'Extraversion_score', 'Agreeableness_score', 'Neuroticism_score']
Moderator: Humanlikeness_score
Controls: ['sentiment_score']
결측 제거 전: 9459
결측 제거 후: 9457


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

                            OLS Regression Results                            
Dep. Variable:             rating_num   R-squared:                       0.046
Model:                            OLS   Adj. R-squared:                  0.045
Method:                 Least Squares   F-statistic:                     37.92
Date:                Fri, 09 Jan 2026   Prob (F-statistic):           1.28e-87
Time:                        13:00:37   Log-Likelihood:                -7743.5
No. Observations:                9457   AIC:                         1.551e+04
Df Residuals:                    9444   BIC:                         1.561e+04
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
const   